# Evaluate positions and batches

This notebook exercises the current `LczeroEvaluator` contract on real `python-chess` boards. It uses the repository's tiny deterministic fixture so it runs offline; replace `load_fixture_evaluator()` with `LczeroModel.from_path(...)` or `LczeroModel.from_hf(...)` for a trained lc0-family model. Fixture observations demonstrate API behavior, not chess strength.

In [ ]:
import chess

from examples.decision_analysis_tutorial import load_fixture_evaluator

runtime = load_fixture_evaluator()
boards = [chess.Board(), chess.Board()]
boards[1].push_uci("e2e4")
evaluations = runtime.evaluator.evaluate(boards)
len(evaluations), evaluations.tensors.batch_size

The evaluator preserves the batch while each row exposes chess-aware legal actions. Raw network logits remain in the nested `TensorDict`; user-facing policy probabilities are masked and normalized over legal moves.

In [ ]:
summary = []
for evaluation in evaluations:
    summary.append(
        {
            "turn": "white" if evaluation.position.turn else "black",
            "best_move": evaluation.policy.best_move.uci(),
            "top_three": [(action.move.uci(), round(action.probability, 4)) for action in evaluation.policy.top(3)],
            "value": evaluation.value.value,
        }
    )
summary

Freeze runtime tensors into an immutable, tensor-free record before persisting or comparing evidence. The digest covers canonical versioned JSON.

In [ ]:
record = evaluations[0].record()
assert record.position.board() == boards[0]
assert record.policy[0].move <= record.policy[-1].move
{"schema_version": record.schema_version, "digest": record.digest()}